# RSTPReid training
Attach `hoanggv/tbps-benchmark` in Kaggle and enable GPU. Run one configuration per notebook run. Training uses the IRRA files at the repository root.

In [ ]:
REPO_URL = "https://github.com/lqb464/irra-gallery.git"
BRANCH = "codex/gallery-context-base-parity"
!git clone -q --branch "$BRANCH" --single-branch $REPO_URL /content/gallery-conditioned-tbps
%cd /content/gallery-conditioned-tbps
!pip -q install -r requirements.txt

In [ ]:
DATA_ROOT = "/content/datasets/benchmark"  # Colab: extracted RSTPReid benchmark root
OUT = "/content/output/rstpreid_run"

## Train IRRA + Gallery Context for 10 epochs

The validation table reports raw CLIP features (`t2i-base`) and contextualized gallery features (`t2i-hard-contrastive`) side by side.

In [ ]:
GALLERY_FLAG = "--gallery_context"
!python /content/gallery-conditioned-tbps/train.py --dataset_name RSTPReid --root_dir "$DATA_ROOT" --output_dir "$OUT" --name irra_gallery --val_dataset val --loss_names sdm+id+mlm --sampler identity --num_instance 4 --batch_size 32 --test_batch_size 256 --num_workers 2 --num_epoch 10 --img_aug --lr 1e-5 --weight_decay 4e-5 --warmup_epochs 5 --lrscheduler cosine --eval_period 1 --seed 17 --MLM $GALLERY_FLAG

## Test the best validation checkpoint

In [ ]:
from pathlib import Path
configs = list(Path(OUT, "RSTPReid").glob("*/configs.yaml"))
if not configs:
    print("Training did not produce a config; fix the training error before evaluation.")
else:
    config = configs[-1]
    !python /content/gallery-conditioned-tbps/test.py --config_file "$config"